# 00 — STL10 unlabeled 데이터 로드 확인

**목표**: torchvision.datasets.STL10이 정상 다운로드/로드되는지, two-view augmentation이 의도대로 작동하는지 확인.

이 노트북은 GPU 사용 안 함. CPU만으로 검증.

## Cell 1 — Import

In [ ]:
%load_ext autoreload
%autoreload 2

import yaml
import torch
import matplotlib.pyplot as plt

from ssl_lib.data import build_stl10_loader, build_train_transform
from ssl_lib.data.stl10_unlabeled import STL10UnlabeledDataset

## Cell 2 — Config 로드

In [ ]:
with open('../configs/mocov2_r50.yaml') as f:
    cfg = yaml.safe_load(f)

# 데이터 디버깅용으로 작은 batch / worker 수로 변경
cfg['training']['batch_size'] = 4
cfg['data']['num_workers'] = 0
cfg['data']['persistent_workers'] = False
cfg['data']['root'] = '../data'
print(yaml.dump(cfg['data'], allow_unicode=True))

## Cell 3 — 데이터셋 빌드 (다운로드 포함)

처음 실행 시 STL10 unlabeled (약 2.6GB) 다운로드.

In [ ]:
transform = build_train_transform(cfg)
dataset = STL10UnlabeledDataset(
    root=cfg['data']['root'],
    transform=transform,
    download=True,
)
print(f'Dataset size: {len(dataset)}')  # 100,000

# 첫 sample 형태 확인
v1, v2 = dataset[0]
print(f'view1 shape: {v1.shape}, dtype: {v1.dtype}, range: [{v1.min():.3f}, {v1.max():.3f}]')
print(f'view2 shape: {v2.shape}, dtype: {v2.dtype}')

## Cell 4 — DataLoader 빌드 + 한 batch 가져오기

In [ ]:
loader = build_stl10_loader(cfg)
v1_batch, v2_batch = next(iter(loader))
print(f'view1 batch: {v1_batch.shape}')  # (4, 3, 96, 96)
print(f'view2 batch: {v2_batch.shape}')

## Cell 5 — 시각화 (두 view 비교)

두 view가 같은 이미지에서 나왔지만 서로 다르게 augmented된 것을 확인.

In [ ]:
from ssl_lib.data.transforms import IMAGENET_MEAN, IMAGENET_STD

def denorm(x):
    """normalize 역연산"""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (x * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes[0, i].imshow(denorm(v1_batch[i]).permute(1, 2, 0))
    axes[0, i].set_title(f'view1 #{i}')
    axes[0, i].axis('off')
    axes[1, i].imshow(denorm(v2_batch[i]).permute(1, 2, 0))
    axes[1, i].set_title(f'view2 #{i}')
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

## ✅ 체크리스트

- [ ] dataset size = 100,000
- [ ] view shape = (3, 96, 96)
- [ ] 두 view가 서로 다른 augmentation 결과로 보임
- [ ] 시각화에서 crop/color/blur 등이 적용된 게 보임